# apriltag分拣与堆叠

### 导入头文件 Import head file

In [1]:
#!/usr/bin/env python3
# coding: utf-8
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from apriltag_identify import ApriltagIdentify
from jetcobot_utils.grasp_controller import GraspController
from jetcobot_utils.jetcobot_config import *

### 初始化机械臂位置 Initialize robot position

In [2]:
apriltagIdentify = ApriltagIdentify()
calibration = Arm_Calibration()
graspController = GraspController()
graspController.init_watch_pose()
dp = []
joint1456 = [39, -71, -8, -8]
msg = {}
threshold = 140
model = "General"
# XYT参数路径  XYT Parameter path
XYT_path="/home/jetson/jetcobot_ws/src/jetcobot_color_identify/scripts/XYT_config.txt"
try: joint1456, threshold = read_XYT(XYT_path)
except Exception: print("Read XYT_config Error !!!")

### 创建控件 Creating widget

In [3]:
button_layout = widgets.Layout(width='320px', height='60px', align_self='center')
output = widgets.Output()
# 调整滑杆 Adjust the slider
joint1_slider      = widgets.IntSlider(description='joint1 :'   ,    value=joint1456[0]     , min=0 , max=90, step=1, orientation='horizontal')
joint4_slider      = widgets.IntSlider(description='joint4 :'   ,    value=joint1456[1]     , min=-110, max=-50, step=1, orientation='horizontal')
joint5_slider      = widgets.IntSlider(description='joint5 :'   ,    value=joint1456[2]     , min=-40, max=30, step=1, orientation='horizontal')
joint6_slider      = widgets.IntSlider(description='joint6 :'   ,    value=joint1456[3]     , min=-40, max=30, step=1, orientation='horizontal')
threshold_slider   = widgets.IntSlider(description='threshold :',    value=threshold , min=0  , max=255, step=1, orientation='horizontal')

# 进入标定模式  Enter calibration mode
position_model     = widgets.Button(description='position_model',  button_style='primary', layout=button_layout)
calibration_model  = widgets.Button(description='calibration_model',  button_style='primary', layout=button_layout)
calibration_ok     = widgets.Button(description='calibration_ok',     button_style='success', layout=button_layout)
calibration_cancel = widgets.Button(description='calibration_cancel', button_style='danger', layout=button_layout)
# 目标检测抓取  Target detection and capture
target_detection   = widgets.Button(description='target_detection', button_style='info', layout=button_layout)
grap_sorting = widgets.Button(description='grap sorting', button_style='success', layout=button_layout)
grap_stacking = widgets.Button(description='grap stacking', button_style='success', layout=button_layout)
# 退出  exit
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
color_img = widgets.VBox([imgbox], layout=widgets.Layout(align_self='center'));
color_identify = widgets.VBox(
    [joint1_slider, joint4_slider, joint5_slider, joint6_slider, threshold_slider, position_model, 
     calibration_model, calibration_ok, calibration_cancel,target_detection, grap_sorting, grap_stacking, exit_button],
    layout=widgets.Layout(align_self='center'));
controls_box = widgets.HBox([color_img, color_identify], layout=widgets.Layout(align_self='center'))

### 标定回调 Calibration callback function

In [4]:
def position_model_Callback(value):
    global model
    model = 'Position'
    with output: print(model)
def calibration_model_Callback(value):
    global model
    model = 'Calibration'
    with output: print(model)
def calibration_OK_Callback(value):
    global model
    model = 'calibration_OK'
    with output: print(model)
def calibration_cancel_Callback(value):
    global model
    model = 'calibration_Cancel'
    with output: print(model)
position_model.on_click(position_model_Callback)
calibration_model.on_click(calibration_model_Callback)
calibration_ok.on_click(calibration_OK_Callback)
calibration_cancel.on_click(calibration_cancel_Callback)

### 模式切换   switching mode

In [5]:
def target_detection_Callback(value):
    global model
    model = 'Detection'
    with output: print(model)
def reset_color_list_Callback(value):
    global model
    model = 'Reset_list'
    with output: print(model)
def grap_sorting_callback(value):
    global model
    model = 'grap_sorting'
    with output: print(model)
def grap_stacking_callback(value):
    global model
    model = 'grap_stacking'
    with output: print(model)
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output: print(model)
target_detection.on_click(target_detection_Callback)
grap_sorting.on_click(grap_sorting_callback)
grap_stacking.on_click(grap_stacking_callback)
exit_button.on_click(exit_button_Callback)

### 主程序 Main process

In [6]:
def camera():
    global model,dp,msg
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0)
    index=1
    # Be executed in loop when the camera is opened normally 
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            _, img = capture.read()
            img = cv.resize(img, (640, 480))
            joint1456=[joint1_slider.value,joint4_slider.value,joint5_slider.value,joint6_slider.value]
            if model == 'Position':
                # 将机械臂移动到标定方框的状态
                joints_angles = [joint1456[0], 0, 0,
                                 joint1456[1], joint1456[2], joint1456[3]]
                print("joints_angles = ",joints_angles)
                graspController.go_calibration_angles(joints_angles)
            if model == 'Calibration':
                # print("model == 'Calibration")
                # 将机械臂移动到标定方框的状态
                joints_angles = [joint1456[0], 0, 0,
                                 joint1456[1], joint1456[2], joint1456[3]]
                # print("joints_angles = ",joints_angles)
                graspController.go_calibration_angles(joints_angles)
                _, img = calibration.calibration_map(img, threshold_slider.value)
            if model == 'calibration_OK':
                try: write_XYT(XYT_path, joint1456, threshold_slider.value)
                except Exception: print("File XYT_config Error !!!")
                dp, img = calibration.calibration_map(img, threshold_slider.value)
                model="General"
            if len(dp) != 0: img = calibration.Perspective_transform(dp, img)
            if model == 'calibration_Cancel':  
                dp = []
                msg= {}
                model="General"
            if len(dp)!= 0 and model == 'Detection':
                img, msg = apriltagIdentify.getApriltagPosMsg(img)
            if model=="Reset_list":
                msg={}
                color_list = {}
                model="General"
            
            if len(msg)!= 0:
                if model == 'grap_sorting':
                    threading.Thread(target=graspController.grasp_run, args=("sorting","apriltag", msg, joint1456)).start()              
                    msg={}
                    model="Detection"
                elif model == 'grap_stacking':
                    threading.Thread(target=graspController.grasp_run, args=("stacking","apriltag", msg, joint1456)).start()  
                    msg={}
                    model="Detection" 
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            index+=1
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

### 启动  Start

In [7]:
# display(controls_box)
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()

[ WARN:0@7.303] global cap_gstreamer.cpp:1777 open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles =  [43, 0, 0, -83, -6, -1]
joints_angles = 